In [1]:
"""
将CB，DNN，tabPFN对应的模型结果画图
"""

import os
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

IN_PATH = "/data2/zhoujb/project/cropVarCefComb/processData/9-1-runOnlineScience/"

# ==============================
# 读取数据
# ==============================
catboost_df = pd.read_csv(os.path.join(IN_PATH, "catboost_compare_results.csv"))
pytorch_df = pd.read_csv(os.path.join(IN_PATH, "pytorch_compare_results.csv"))
tabpfn_df  = pd.read_csv(os.path.join(IN_PATH, "tabpfn_compare_results.csv"))
# 将 TabPFN 的列名重命名为 Dynamic，保证后续 to_long_result 可以直接使用
tabpfn_df = tabpfn_df.rename(columns={"TrainingOnebyOne_PCC": "Dynamic"})

In [8]:
# ==============================
# 预处理函数
# ==============================
def preprocess_df(df):
    """
    仅保留所需列。
    如果同一个 Trait/year/location 有重复记录，
    对 PCC 列取平均值
    """
    # 保留可用列
    cols = [col for col in df.columns if col in ['Trait','region', 'TraingWhole_PCC','TraingOnebyOne_PCC','TraingTwice_PCC','Dynamic']]
    tmp = df[cols].copy()
    
    # 如果存在 TraingWhole/OnebyOne/Twice，则对相应列取平均
    pcc_cols = [c for c in tmp.columns if '_PCC' in c or c=='Dynamic']
    if pcc_cols:
        tmp = tmp.groupby(['Trait', 'region'], as_index=False)[pcc_cols].mean()
    
    return tmp

# ==============================
# 长表转换函数
# ==============================
def to_long_result(df, model_name, dynamic_only=False):
    """
    df -> 长表
    dynamic_only=True 表示只保留 Dynamic 方法
    """
    tmp = preprocess_df(df)
    
    # 重命名列
    rename_dict = {}
    if 'TraingWhole_PCC' in tmp.columns:
        rename_dict['TraingWhole_PCC'] = 'FitAll'
    if 'TraingOnebyOne_PCC' in tmp.columns:
        rename_dict['TraingOnebyOne_PCC'] = 'FitOne'
    if 'TraingTwice_PCC' in tmp.columns:
        rename_dict['TraingTwice_PCC'] = 'Dynamic'
    if 'Dynamic' in tmp.columns:
        rename_dict['Dynamic'] = 'Dynamic'
    
    tmp = tmp.rename(columns=rename_dict)
    
    # 如果只保留 Dynamic 方法
    value_vars = ['FitAll','FitOne','Dynamic']
    value_vars = [v for v in value_vars if v in tmp.columns]
    if dynamic_only:
        value_vars = ['Dynamic']
    
    # 宽表转长表
    tmp_long = tmp.melt(
        id_vars=['Trait','region'],
        value_vars=value_vars,
        var_name='Method',
        value_name='PCC'
    )
    
    tmp_long['Model'] = model_name
    tmp_long['Model_Method'] = tmp_long['Model'] + '_' + tmp_long['Method']
    
    tmp_long = tmp_long[['Trait','region', 'Model','Method','PCC','Model_Method']]
    
    return tmp_long

# ==============================
# 转换各模型为长表
# ==============================
cb_long = to_long_result(catboost_df, 'CB')
pt_long = to_long_result(pytorch_df, 'DNN')
tp_long = to_long_result(tabpfn_df, 'TabPFN', dynamic_only=True)

# 合并
merged_long_df = pd.concat([cb_long, pt_long, tp_long], ignore_index=True)


# 排序
merged_long_df = merged_long_df.sort_values(
    by=['Trait','region', 'Model','Method']
).reset_index(drop=True)

merged_long_df.to_csv(os.path.join(IN_PATH, "all_method_pcc_res.csv"), index=0)

In [9]:
merged_long_df.head()

,Trait,region,Model,Method,PCC,Model_Method
0,Culm length,Hainan,CB,Dynamic,0.999154,CB_Dynamic
1,Culm length,Hainan,CB,FitAll,0.999035,CB_FitAll
2,Culm length,Hainan,CB,FitOne,0.998977,CB_FitOne
3,Culm length,Hainan,DNN,Dynamic,0.998684,DNN_Dynamic
4,Culm length,Hainan,DNN,FitAll,0.999001,DNN_FitAll


In [10]:
# ==============================
# 横坐标顺序与配色
# ==============================

method_order = [
    'CB_FitAll', 'CB_FitOne', 'CB_Dynamic',
    'DNN_FitAll', 'DNN_FitOne', 'DNN_Dynamic',
    'TabPFN_Dynamic'
]

merged_long_df["Model_Method"] = merged_long_df["Model_Method"].astype("category")
merged_long_df["Model_Method"] = merged_long_df["Model_Method"].cat.set_categories(
    method_order,
    ordered=True
)

palette = {
    'CB_FitAll':  '#1f77b4',
    'CB_FitOne':  '#4c9ed9',
    'CB_Dynamic': '#08306b',
    'DNN_FitAll': '#d62728',
    'DNN_FitOne': '#ff9896',
    'DNN_Dynamic':'#7f0000',
    'TabPFN_Dynamic':'#2ca02c'
}


# ==============================
# 检查必要列
# ==============================

required_cols = ["Trait", "region", "Model_Method", "PCC"]

for col in required_cols:
    if col not in merged_long_df.columns:
        raise ValueError(f"merged_long_df 中缺少必要列: {col}")


# ==============================
# region 顺序
# ==============================
# 如果你希望固定 region 顺序，可以在这里手动指定，例如：
# region_order = ["Region1", "Region2", "Region3"]
#
# 当前代码默认按照 region 名称排序。

merged_long_df["region"] = merged_long_df["region"].astype(str)

region_order = sorted(
    merged_long_df["region"].dropna().unique()
)


# ==============================
# 输出目录
# ==============================

outdir = os.path.join(IN_PATH, "trait_relplot")
os.makedirs(outdir, exist_ok=True)

sns.set(font_scale=1.2, style="ticks")


# ==============================
# 作图
# ==============================

for trait in merged_long_df["Trait"].dropna().unique():

    df_trait = merged_long_df[
        merged_long_df["Trait"] == trait
    ].copy()

    # 只保留有 PCC 数据的行
    df_trait = df_trait.dropna(
        subset=["PCC", "Model_Method", "region"]
    )

    # 如果当前性状没有任何可画数据，则跳过
    if len(df_trait) == 0:
        print(f"Skip trait {trait}: no valid PCC data.")
        continue

    # ==============================
    # 每个性状画一张图
    # 每张图一行，三个 region 作为三个子图
    # ==============================

    g = sns.relplot(
        data=df_trait,
        x="Model_Method",
        y="PCC",
        kind="scatter",
        marker="o",
        s=80,

        # 原来是 row="location", col="year"
        # 现在只有 region，因此只用 col="region"
        col="region",
        col_order=region_order,

        height=4.5,
        aspect=1.2,

        facet_kws=dict(
            sharey=False,
            sharex=False
        ),

        hue="Model_Method",
        palette=palette,
        hue_order=method_order
    )

    # 去掉 legend
    if g._legend is not None:
        g._legend.remove()

    # 设置子图标题
    g.set_titles(
        col_template="Region = {col_name}"
    )

    # ==============================
    # 调整每个 region 子图
    # ==============================

    col_vals = g.col_names

    for ax, region in zip(g.axes.flat, col_vals):

        sub = df_trait[
            df_trait["region"] == str(region)
        ].copy()

        if len(sub) == 0:
            # 如果某个 region 没有数据，则隐藏该子图
            ax.set_visible(False)
            continue

        ymin = sub["PCC"].min()
        ymax = sub["PCC"].max()

        if pd.isna(ymin) or pd.isna(ymax):
            ax.set_visible(False)
            continue

        if ymin == ymax:
            ymin -= 0.02
            ymax += 0.02
        else:
            margin = (ymax - ymin) * 0.15
            ymin -= margin
            ymax += margin

        ax.set_ylim(ymin, ymax)

        # 每个子图横坐标都显示所有模型方法
        ax.set_xticks(range(len(method_order)))
        ax.set_xticklabels(
            method_order,
            rotation=45,
            ha="right"
        )

        ax.set_xlabel("")
        ax.set_ylabel("PCC")

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    plt.tight_layout()

    # ==============================
    # 保存图片
    # ==============================

    safe_trait = (
        str(trait)
        .replace("/", "_")
        .replace(" ", "_")
        .replace("\\", "_")
        .replace(":", "_")
    )

    pdf_file = os.path.join(
        outdir,
        f"{safe_trait}_PCC_region_relplot.pdf"
    )

    png_file = os.path.join(
        outdir,
        f"{safe_trait}_PCC_region_relplot.png"
    )

    g.fig.savefig(
        pdf_file,
        format="pdf",
        bbox_inches="tight",
        transparent=True
    )


    plt.close(g.fig)

    print(f"Saved: {pdf_file}")
    print(f"Saved: {png_file}")


print("All figures saved in:", outdir)

/data2/zhoujb/anaconda3/envs/scRNAML/lib/python3.11/site-packages/seaborn/axisgrid.py:118: UserWarning: The figure layout has changed to tight
  self._figure.tight_layout(*args, **kwargs)
/tmp/ipykernel_390339/2451614785.py:172: UserWarning: The figure layout has changed to tight
  plt.tight_layout()


Saved: /data2/zhoujb/project/cropVarCefComb/processData/9-1-runOnlineScience/trait_relplot/Culm_length_PCC_region_relplot.pdf
Saved: /data2/zhoujb/project/cropVarCefComb/processData/9-1-runOnlineScience/trait_relplot/Culm_length_PCC_region_relplot.png


/data2/zhoujb/anaconda3/envs/scRNAML/lib/python3.11/site-packages/seaborn/axisgrid.py:118: UserWarning: The figure layout has changed to tight
  self._figure.tight_layout(*args, **kwargs)
/tmp/ipykernel_390339/2451614785.py:172: UserWarning: The figure layout has changed to tight
  plt.tight_layout()


Saved: /data2/zhoujb/project/cropVarCefComb/processData/9-1-runOnlineScience/trait_relplot/Heading_date_PCC_region_relplot.pdf
Saved: /data2/zhoujb/project/cropVarCefComb/processData/9-1-runOnlineScience/trait_relplot/Heading_date_PCC_region_relplot.png


/data2/zhoujb/anaconda3/envs/scRNAML/lib/python3.11/site-packages/seaborn/axisgrid.py:118: UserWarning: The figure layout has changed to tight
  self._figure.tight_layout(*args, **kwargs)
/tmp/ipykernel_390339/2451614785.py:172: UserWarning: The figure layout has changed to tight
  plt.tight_layout()


...

Saved: /data2/zhoujb/project/cropVarCefComb/processData/9-1-runOnlineScience/trait_relplot/Panicle_length_PCC_region_relplot.pdf
Saved: /data2/zhoujb/project/cropVarCefComb/processData/9-1-runOnlineScience/trait_relplot/Panicle_length_PCC_region_relplot.png


/data2/zhoujb/anaconda3/envs/scRNAML/lib/python3.11/site-packages/seaborn/axisgrid.py:118: UserWarning: The figure layout has changed to tight
  self._figure.tight_layout(*args, **kwargs)
/tmp/ipykernel_390339/2451614785.py:172: UserWarning: The figure layout has changed to tight
  plt.tight_layout()


Saved: /data2/zhoujb/project/cropVarCefComb/processData/9-1-runOnlineScience/trait_relplot/Panicle_number_PCC_region_relplot.pdf
Saved: /data2/zhoujb/project/cropVarCefComb/processData/9-1-runOnlineScience/trait_relplot/Panicle_number_PCC_region_relplot.png


/data2/zhoujb/anaconda3/envs/scRNAML/lib/python3.11/site-packages/seaborn/axisgrid.py:118: UserWarning: The figure layout has changed to tight
  self._figure.tight_layout(*args, **kwargs)
/tmp/ipykernel_390339/2451614785.py:172: UserWarning: The figure layout has changed to tight
  plt.tight_layout()


Saved: /data2/zhoujb/project/cropVarCefComb/processData/9-1-runOnlineScience/trait_relplot/Plant_height_PCC_region_relplot.pdf
Saved: /data2/zhoujb/project/cropVarCefComb/processData/9-1-runOnlineScience/trait_relplot/Plant_height_PCC_region_relplot.png
All figures saved in: /data2/zhoujb/project/cropVarCefComb/processData/9-1-runOnlineScience/trait_relplot
